In [4]:
# ============================================================
# Reddit Startups Content Analysis
# Stage 1: Import Libraries & Load Data
# ============================================================

import pandas as pd
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('/Users/ssnkeerthana/Desktop/reddit-startups-content-analysis/data/raw/reddit_database.csv')

# First look at the data
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)

Shape: (545427, 13)

Columns: ['created_date', 'created_timestamp', 'subreddit', 'title', 'id', 'author', 'author_created_utc', 'full_link', 'score', 'num_comments', 'num_crossposts', 'subreddit_subscribers', 'post']

First 3 rows:


,created_date,created_timestamp,subreddit,title,id,author,author_created_utc,full_link,score,num_comments,num_crossposts,subreddit_subscribers,post
0,2010-02-10 22:06:17,1.265832e+09,analytics,YouTube's traffic data for music questioned,b0ih7,salvage,1.184143e+09,https://www.reddit.com/r/analytics/comments/b0...,3.0,0.0,0.0,NaN,NaN
1,2010-02-10 22:06:53,1.265832e+09,analytics,November Sees Number of U.S. Videos Viewed Onl...,b0ihf,salvage,1.184143e+09,https://www.reddit.com/r/analytics/comments/b0...,1.0,0.0,0.0,NaN,NaN
2,2010-02-11 19:47:22,1.265910e+09,analytics,So what do you guys all do related to analytic...,b0x63,xtom,1.227476e+09,https://www.reddit.com/r/analytics/comments/b0...,7.0,4.0,0.0,NaN,There's a lot of reasons to want to know all t...


In [7]:
# ============================================================
# Stage 2: Initial Data Exploration
# ============================================================

# How many rows and columns?
print("Shape:", df.shape)

# What are the data types?
print("\nData Types:")
print(df.dtypes)

# How many missing values in each column?
print("\nMissing Values:")
print(df.isnull().sum())

# Basic stats
print("\nBasic Stats:")
print(df[['score', 'num_comments', 'num_crossposts']].describe())

Shape: (545427, 13)

Data Types:
created_date              object
created_timestamp        float64
subreddit                 object
title                     object
id                        object
author                    object
author_created_utc       float64
full_link                 object
score                    float64
num_comments             float64
num_crossposts           float64
subreddit_subscribers    float64
post                      object
dtype: object

Missing Values:
created_date                  0
created_timestamp             0
subreddit                     0
title                         0
id                            0
author                        0
author_created_utc       453442
full_link                     0
score                         0
num_comments                  0
num_crossposts           112425
subreddit_subscribers    139617
post                     271218
dtype: int64

Basic Stats:
               score   num_comments  num_crossposts
count  54542

In [10]:
# ============================================================
# Stage 3: Data Cleaning
# ============================================================

# Step 1: Drop columns we don't need for analysis
df = df.drop(columns=['author_created_utc', 'num_crossposts', 'id'])

# Step 2: Remove duplicate posts
print("Before removing duplicates:", df.shape)
df = df.drop_duplicates(subset=['title', 'author'])
print("After removing duplicates:", df.shape)

# Step 3: Remove deleted and bot authors
df = df[~df['author'].isin(['[deleted]', 'AutoModerator'])]
print("After removing deleted/bot authors:", df.shape)

# Step 4: Remove rows where title is missing
df = df.dropna(subset=['title'])
print("After dropping missing titles:", df.shape)

# Step 5: Fix impossible values — num_comments cannot be negative
df = df[df['num_comments'] >= 0]
print("After fixing negative comments:", df.shape)

Before removing duplicates: (545427, 10)
After removing duplicates: (500133, 10)
After removing deleted/bot authors: (479157, 10)
After dropping missing titles: (479157, 10)
After fixing negative comments: (479156, 10)


In [14]:
# ============================================================
# Step 6: Convert created_date to datetime
# and extract useful time features
# ============================================================

# Convert to datetime
df['created_date'] = pd.to_datetime(df['created_date'])

# Extract time features
df['hour'] = df['created_date'].dt.hour
df['day_of_week'] = df['created_date'].dt.day_name()
df['month'] = df['created_date'].dt.month_name()
df['year'] = df['created_date'].dt.year

# Verify
print("New columns added:")
print(df[['created_date', 'hour', 'day_of_week', 'month', 'year']].head())

New columns added:
         created_date  hour day_of_week     month  year
0 2010-02-10 22:06:17    22   Wednesday  February  2010
1 2010-02-10 22:06:53    22   Wednesday  February  2010
2 2010-02-11 19:47:22    19    Thursday  February  2010
5 2010-03-04 20:17:26    20    Thursday     March  2010
6 2010-03-21 10:51:48    10      Sunday     March  2010


In [17]:
# ============================================================
# Step 7: Clean the title column
# and add engagement features
# ============================================================

# Clean title — strip extra whitespace
df['title'] = df['title'].str.strip()

# Add title length as a feature
df['title_length'] = df['title'].str.len()

# Add engagement score — combines upvotes and comments
df['engagement_score'] = df['score'] + df['num_comments']

# Fill missing post body with empty string
df['post'] = df['post'].fillna('')

# Fill missing subreddit_subscribers with 0
df['subreddit_subscribers'] = df['subreddit_subscribers'].fillna(0)

# Verify
print("Final cleaned dataset shape:", df.shape)
print("\nNew features added:")
print(df[['title', 'title_length', 'score', 'num_comments', 'engagement_score']].head())

Final cleaned dataset shape: (479156, 16)

New features added:
                                               title  title_length  score  \
0        YouTube's traffic data for music questioned            43    3.0   
1  November Sees Number of U.S. Videos Viewed Onl...           104    1.0   
2  So what do you guys all do related to analytic...            66    7.0   
5  Google's Invasive, non-Anonymized Ad Targeting...           107    2.0   
6       700 Million Monthly YouTube Visitors by 2016            44    2.0   

   num_comments  engagement_score  
0           0.0               3.0  
1           0.0               1.0  
2           4.0              11.0  
5           1.0               3.0  
6           0.0               2.0  


In [20]:
# ============================================================
# Step 8: Save cleaned dataset
# ============================================================

cleaned_path = '/Users/ssnkeerthana/Desktop/reddit-startups-content-analysis/data/cleaned/reddit_cleaned.csv'

df.to_csv(cleaned_path, index=False)

print("✅ Cleaned dataset saved successfully!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


✅ Cleaned dataset saved successfully!
Shape: (479156, 16)
Columns: ['created_date', 'created_timestamp', 'subreddit', 'title', 'author', 'full_link', 'score', 'num_comments', 'subreddit_subscribers', 'post', 'hour', 'day_of_week', 'month', 'year', 'title_length', 'engagement_score']


In [23]:
# ============================================================
# Stage 3: Load Cleaned Data into SQLite
# ============================================================

import sqlite3

# Connect to SQLite database
conn = sqlite3.connect('/Users/ssnkeerthana/Desktop/reddit-startups-content-analysis/reddit_startups.db')

# Load cleaned data into SQLite
df.to_sql('reddit_posts', conn, if_exists='replace', index=False)

print("✅ Data loaded into SQLite successfully!")
print("Total rows loaded:", len(df))

✅ Data loaded into SQLite successfully!
Total rows loaded: 479156


In [26]:
# ============================================================
# Stage 4: SQL Analysis
# Query 1: Top 10 subreddits by average engagement
# ============================================================

query1 = """
SELECT 
    subreddit,
    COUNT(*) AS total_posts,
    ROUND(AVG(score), 2) AS avg_score,
    ROUND(AVG(num_comments), 2) AS avg_comments,
    ROUND(AVG(engagement_score), 2) AS avg_engagement
FROM reddit_posts
GROUP BY subreddit
HAVING total_posts > 100
ORDER BY avg_engagement DESC
LIMIT 10;
"""

result1 = pd.read_sql_query(query1, conn)
print("Top 10 Subreddits by Engagement:")
print(result1)

Top 10 Subreddits by Engagement:
              subreddit  total_posts  avg_score  avg_comments  avg_engagement
0       MachineLearning       104135       7.25          4.61           11.87
1           datascience        61515       4.53          5.98           10.51
2                rstats        14043       4.25          5.13            9.38
3            artificial        28169       5.61          3.19            8.80
4            statistics        55327       3.08          4.56            7.64
5       dataengineering        10755       1.64          5.76            7.40
6       computerscience        40614       3.29          3.81            7.10
7              datasets        15729       4.42          2.47            6.90
8  learnmachinelearning        38853       3.77          2.82            6.59
9        computervision        12656       2.94          3.47            6.41


In [29]:
# ============================================================
# Query 2: Best day of week to post for highest engagement
# ============================================================

query2 = """
SELECT 
    day_of_week,
    COUNT(*) AS total_posts,
    ROUND(AVG(engagement_score), 2) AS avg_engagement,
    ROUND(AVG(score), 2) AS avg_score,
    ROUND(AVG(num_comments), 2) AS avg_comments
FROM reddit_posts
GROUP BY day_of_week
ORDER BY avg_engagement DESC;
"""

result2 = pd.read_sql_query(query2, conn)
print("Best Days to Post:")
print(result2)

Best Days to Post:
  day_of_week  total_posts  avg_engagement  avg_score  avg_comments
0      Sunday        52181            9.00       4.42          4.58
1    Saturday        53588            8.43       4.11          4.32
2   Wednesday        78298            8.22       4.31          3.91
3     Tuesday        76893            8.06       4.17          3.90
4      Friday        70114            8.03       4.08          3.95
5      Monday        71505            7.97       4.13          3.83
6    Thursday        76577            7.75       3.93          3.82


In [32]:
# ============================================================
# Query 3: Best hour of day to post for highest engagement
# ============================================================

query3 = """
SELECT 
    hour,
    COUNT(*) AS total_posts,
    ROUND(AVG(engagement_score), 2) AS avg_engagement,
    ROUND(AVG(score), 2) AS avg_score,
    ROUND(AVG(num_comments), 2) AS avg_comments
FROM reddit_posts
GROUP BY hour
ORDER BY avg_engagement DESC
LIMIT 10;
"""

result3 = pd.read_sql_query(query3, conn)
print("Best Hours to Post (Top 10):")
print(result3)

Best Hours to Post (Top 10):
   hour  total_posts  avg_engagement  avg_score  avg_comments
0    17        27116            9.11       4.92          4.19
1     4        15526            8.91       4.65          4.26
2     2        16538            8.83       4.39          4.44
3    19        27345            8.69       4.46          4.23
4    15        21867            8.65       4.42          4.23
5    16        24562            8.47       4.28          4.18
6     1        18236            8.43       4.09          4.34
7    21        26531            8.37       4.24          4.13
8    18        28403            8.37       4.36          4.01
9    23        22791            8.28       4.21          4.07


In [35]:
# ============================================================
# Query 4: Does title length affect engagement?
# ============================================================

query4 = """
SELECT 
    CASE 
        WHEN title_length < 30 THEN 'Short (< 30 chars)'
        WHEN title_length BETWEEN 30 AND 60 THEN 'Medium (30-60 chars)'
        WHEN title_length BETWEEN 61 AND 100 THEN 'Long (61-100 chars)'
        ELSE 'Very Long (> 100 chars)'
    END AS title_category,
    COUNT(*) AS total_posts,
    ROUND(AVG(engagement_score), 2) AS avg_engagement,
    ROUND(AVG(score), 2) AS avg_score,
    ROUND(AVG(num_comments), 2) AS avg_comments
FROM reddit_posts
GROUP BY title_category
ORDER BY avg_engagement DESC;
"""

result4 = pd.read_sql_query(query4, conn)
print("Title Length vs Engagement:")
print(result4)


Title Length vs Engagement:
            title_category  total_posts  avg_engagement  avg_score  \
0  Very Long (> 100 chars)        47944           10.44       5.35   
1      Long (61-100 chars)       145483            8.95       4.76   
2     Medium (30-60 chars)       216613            7.58       3.79   
3       Short (< 30 chars)        69116            6.75       3.19   

   avg_comments  
0          5.08  
1          4.19  
2          3.79  
3          3.55  


In [38]:
# ============================================================
# Stage 5: Export Results for Power BI Dashboard
# ============================================================

base_path = '/Users/ssnkeerthana/Desktop/reddit-startups-content-analysis/data/cleaned/'

result1.to_csv(base_path + 'top_subreddits.csv', index=False)
result2.to_csv(base_path + 'best_days.csv', index=False)
result3.to_csv(base_path + 'best_hours.csv', index=False)
result4.to_csv(base_path + 'title_length_engagement.csv', index=False)

# Close database connection
conn.close()

print("✅ All query results exported successfully!")
print("Files saved:")
print("  - top_subreddits.csv")
print("  - best_days.csv")
print("  - best_hours.csv")
print("  - title_length_engagement.csv")

✅ All query results exported successfully!
Files saved:
  - top_subreddits.csv
  - best_days.csv
  - best_hours.csv
  - title_length_engagement.csv


In [40]:
# Preview the cleaned dataset
df.head(10)

,created_date,created_timestamp,subreddit,title,author,full_link,score,num_comments,subreddit_subscribers,post,hour,day_of_week,month,year,title_length,engagement_score
0,2010-02-10 22:06:17,1.265832e+09,analytics,YouTube's traffic data for music questioned,salvage,https://www.reddit.com/r/analytics/comments/b0...,3.0,0.0,0.0,,22,Wednesday,February,2010,43,3.0
1,2010-02-10 22:06:53,1.265832e+09,analytics,November Sees Number of U.S. Videos Viewed Onl...,salvage,https://www.reddit.com/r/analytics/comments/b0...,1.0,0.0,0.0,,22,Wednesday,February,2010,104,1.0
2,2010-02-11 19:47:22,1.265910e+09,analytics,So what do you guys all do related to analytic...,xtom,https://www.reddit.com/r/analytics/comments/b0...,7.0,4.0,0.0,There's a lot of reasons to want to know all t...,19,Thursday,February,2010,66,11.0
5,2010-03-04 20:17:26,1.267727e+09,analytics,"Google's Invasive, non-Anonymized Ad Targeting...",xtom,https://www.reddit.com/r/analytics/comments/b9...,2.0,1.0,0.0,"I'm cross posting this from /r/cyberlaw, hopef...",20,Thursday,March,2010,107,3.0
6,2010-03-21 10:51:48,1.269162e+09,analytics,700 Million Monthly YouTube Visitors by 2016,salvage,https://www.reddit.com/r/analytics/comments/bg...,2.0,0.0,0.0,,10,Sunday,March,2010,44,2.0
7,2010-03-24 03:55:43,1.269396e+09,analytics,Best Free Web Analytics Tools for Your Websites,wpconstructs,https://www.reddit.com/r/analytics/comments/bh...,2.0,0.0,0.0,,3,Wednesday,March,2010,47,2.0
9,2010-04-06 23:14:05,1.270585e+09,analytics,Google Search Funnels: The greatest assist spe...,dredman,https://www.reddit.com/r/analytics/comments/bn...,1.0,1.0,0.0,,23,Tuesday,April,2010,73,2.0
10,2010-04-08 13:09:34,1.270721e+09,analytics,How To Analyze Your Site Stats,blondishnet,https://www.reddit.com/r/analytics/comments/bo...,1.0,0.0,0.0,,13,Thursday,April,2010,30,1.0
11,2010-04-08 23:05:18,1.270757e+09,analytics,Tracking Multiple Websites &amp; Blogs in Goog...,lisbongirl,https://www.reddit.com/r/analytics/comments/bo...,7.0,0.0,0.0,,23,Thursday,April,2010,80,7.0
12,2010-04-16 15:46:22,1.271422e+09,analytics,22 Absolutely Free Twitter Web Analytics Tools,kittuk,https://www.reddit.com/r/analytics/comments/br...,2.0,1.0,0.0,,15,Friday,April,2010,46,3.0


In [3]:
# ============================================================
# Combine all 4 datasets into one master CSV for Power BI
# ============================================================

import pandas as pd

base_path = '/Users/ssnkeerthana/Desktop/reddit-startups-content-analysis/data/cleaned/'

# Load all 4 datasets
top_subreddits = pd.read_csv(base_path + 'top_subreddits.csv')
best_days = pd.read_csv(base_path + 'best_days.csv')
best_hours = pd.read_csv(base_path + 'best_hours.csv')
title_length = pd.read_csv(base_path + 'title_length_engagement.csv')

# Add a category column to each
top_subreddits['category'] = 'Subreddit Performance'
best_days['category'] = 'Best Days to Post'
best_hours['category'] = 'Best Hours to Post'
title_length['category'] = 'Title Length Impact'

print("✅ All datasets loaded!")
print("top_subreddits:", top_subreddits.shape)
print("best_days:", best_days.shape)
print("best_hours:", best_hours.shape)
print("title_length:", title_length.shape)

✅ All datasets loaded!
top_subreddits: (10, 6)
best_days: (7, 6)
best_hours: (10, 6)
title_length: (4, 6)


In [6]:
# ============================================================
# Save each dataset with clean professional column names
# ============================================================

# Rename columns professionally
top_subreddits = top_subreddits.rename(columns={
    'subreddit': 'Subreddit',
    'total_posts': 'Total Posts',
    'avg_score': 'Avg Upvotes',
    'avg_comments': 'Avg Comments',
    'avg_engagement': 'Avg Engagement Score',
    'category': 'Category'
})

best_days = best_days.rename(columns={
    'day_of_week': 'Day of Week',
    'total_posts': 'Total Posts',
    'avg_engagement': 'Avg Engagement Score',
    'avg_score': 'Avg Upvotes',
    'avg_comments': 'Avg Comments',
    'category': 'Category'
})

best_hours = best_hours.rename(columns={
    'hour': 'Hour of Day',
    'total_posts': 'Total Posts',
    'avg_engagement': 'Avg Engagement Score',
    'avg_score': 'Avg Upvotes',
    'avg_comments': 'Avg Comments',
    'category': 'Category'
})

title_length = title_length.rename(columns={
    'title_category': 'Title Length Category',
    'total_posts': 'Total Posts',
    'avg_engagement': 'Avg Engagement Score',
    'avg_score': 'Avg Upvotes',
    'avg_comments': 'Avg Comments',
    'category': 'Category'
})

# Save each cleaned file
top_subreddits.to_csv(base_path + 'top_subreddits_final.csv', index=False)
best_days.to_csv(base_path + 'best_days_final.csv', index=False)
best_hours.to_csv(base_path + 'best_hours_final.csv', index=False)
title_length.to_csv(base_path + 'title_length_final.csv', index=False)

print("✅ All files saved with professional column names!")

✅ All files saved with professional column names!
